# Starlink — Step 1: understand the Kepler data

Working concept: StarSignal. The GitHub repository is named Starlink.

**Problem:** Classify labelled Kepler observations as confirmed planets or false positives using selected transit and stellar measurements; explain the results visually.

**Today's goal:** Load a documented snapshot, understand one row, inspect the labels, and identify missing values. We will not train a model in this notebook.

The data are selected columns from the NASA Exoplanet Archive's cumulative KOI table. They are extracted measurements, not raw brightness time series. Some quantities are fitted estimates. Archive labels can change, so the download is pinned with a date and SHA-256 checksum in `data/raw/provenance.json`.

Sources: [Column definitions](https://exoplanetarchive.ipac.caltech.edu/docs/API_kepcandidate_columns.html), [KOI documentation](https://exoplanetarchive.ipac.caltech.edu/docs/Kepler_KOI_docs.html), DOI [10.26133/NEA4](https://doi.org/10.26133/NEA4).

## Before running

Locally, run Jupyter from the extracted project folder. In Google Colab, upload this notebook, then upload `data/raw/kepler_koi.csv` through the Files panel. The code supports either location. Run cells one at a time with Shift+Enter.


## 1. Load the table
`pd` is our short name for Pandas. A **DataFrame** is a table: rows are observations, columns are properties. CSV means comma-separated values.


In [1]:
import pandas as pd
from pathlib import Path

data_path = Path("data/raw/kepler_koi.csv")
if not data_path.exists():
    data_path = Path("kepler_koi.csv")  # Colab upload location
if not data_path.exists():
    raise FileNotFoundError("Upload kepler_koi.csv, or run from the extracted project folder.")

df = pd.read_csv(data_path)
print("Rows and columns:", df.shape)
df.head()


Rows and columns: (9564, 9)


,kepid,kepoi_name,koi_disposition,koi_period,koi_duration,koi_depth,koi_model_snr,koi_steff,koi_srad
0,11446443,K00001.01,CONFIRMED,2.470613,1.74319,14230.9,4304.3,5820.0,0.964
1,10666592,K00002.01,CONFIRMED,2.204735,3.88864,6674.7,5945.9,6440.0,1.952
2,10748390,K00003.01,CONFIRMED,4.887803,2.36255,4315.8,862.3,4778.0,0.763
3,3861595,K00004.01,CONFIRMED,3.849372,2.63200,1298.6,132.7,6244.0,2.992
4,8554498,K00005.01,CANDIDATE,4.780328,2.02530,960.8,380.8,5937.0,1.787


## 2. Identify the columns
| Column | Role | Meaning / unit |
|---|---|---|
| `kepid` | Group identifier | Host-star target ID; keep the same star in one evaluation split |
| `kepoi_name` | Observation identifier | KOI name; not a model input |
| `koi_disposition` | Label | Archived classification; not a model input |
| `koi_period` | Feature | Repetition period, days |
| `koi_duration` | Feature | Transit duration, hours |
| `koi_depth` | Feature | Lost brightness at transit minimum, parts per million |
| `koi_model_snr` | Feature | Transit signal-to-noise ratio, dimensionless |
| `koi_steff` | Feature | Stellar effective temperature, kelvin |
| `koi_srad` | Feature | Stellar radius, in units of the Sun's radius |

Features are inputs. The label is the answer we will learn to predict. Identifiers track records and control splitting; they will not be fed into the classifier.


In [2]:
features = [
    "koi_period", "koi_duration", "koi_depth",
    "koi_model_snr", "koi_steff", "koi_srad"
]
df.dtypes


kepid                int64
kepoi_name          object
koi_disposition     object
koi_period         float64
koi_duration       float64
koi_depth          float64
koi_model_snr      float64
koi_steff          float64
koi_srad           float64
dtype: object

## 3. Read one actual observation
K00752.01 is a confirmed-labelled example. A depth of 615.8 ppm corresponds to 0.06158% lost brightness. Its stellar radius of 0.927 means 92.7% of the Sun's radius. These are properties of this example, not universal rules for planets.


In [3]:
example = df.loc[df["kepoi_name"] == "K00752.01"]
example.T


,1080
kepid,10797460
kepoi_name,K00752.01
koi_disposition,CONFIRMED
koi_period,9.488036
koi_duration,2.9575
koi_depth,615.8
koi_model_snr,35.8
koi_steff,5455.0
koi_srad,0.927


In [4]:
depth_ppm = example["koi_depth"].iloc[0]
depth_percent = depth_ppm / 10_000
print(f"Transit depth: {depth_percent:.5f}%")


Transit depth: 0.06158%


## 4. Count the labels
CONFIRMED and FALSE POSITIVE provide our two supervised-learning classes. CANDIDATE is unresolved for this task and stays outside labelled training. These are archive labels, not Starlink predictions.


In [5]:
df["koi_disposition"].value_counts()


koi_disposition
FALSE POSITIVE    4839
CONFIRMED         2748
CANDIDATE         1977
Name: count, dtype: int64

In [6]:
labelled = df.loc[
    df["koi_disposition"].isin(["CONFIRMED", "FALSE POSITIVE"])
].copy()

labelled["target"] = labelled["koi_disposition"].map({
    "FALSE POSITIVE": 0,
    "CONFIRMED": 1,
})
print("Rows available for later supervised work:", len(labelled))
labelled[["kepoi_name", "koi_disposition", "target"]].head()


Rows available for later supervised work: 7587


,kepoi_name,koi_disposition,target
0,K00001.01,CONFIRMED,1
1,K00002.01,CONFIRMED,1
2,K00003.01,CONFIRMED,1
3,K00004.01,CONFIRMED,1
5,K00005.02,FALSE POSITIVE,0


## 5. Audit missing measurements and repeated stars
`isna()` marks absent values. `sum()` counts them by column. A missing value is not zero. Today we record missingness; later any learned imputation will be fitted only on training data.

Multiple KOIs can share a host star. We will group by `kepid` when splitting the data to avoid having one star represented in both training and evaluation.


In [7]:
labelled[features].isna().sum()


koi_period         0
koi_duration       0
koi_depth        259
koi_model_snr    259
koi_steff        259
koi_srad         259
dtype: int64

In [8]:
print("Repeated observation IDs:", df["kepoi_name"].duplicated().sum())
print("Unique host stars:", df["kepid"].nunique())
print("Labelled rows with an absent measurement:",
      labelled[features].isna().any(axis=1).sum())
print("Labelled host stars:", labelled["kepid"].nunique())


Repeated observation IDs: 0
Unique host stars: 8214
Labelled rows with an absent measurement: 259
Labelled host stars: 6641
